# hypnose-sleap

The pipeline is a command line; this notebook is for poking at one session
interactively. Everything below has a verb:

| notebook call | command |
| --- | --- |
| `infer_session` | `hypnose-sleap infer -s 57 -d 20260717` |
| `extract_session` | `hypnose-sleap extract -s 57 -d 20260717` |
| `combine_session` | `hypnose-sleap combine -s 57 -d 20260717` |
| both | `hypnose-sleap run -s 57 -d 20260717` |
| `annotate_session` | `hypnose-sleap annotate -s 57 -d 20260717 --rotate 90` |
| - | `hypnose-sleap fetch` / `push` |

Paths come from `configs/data_locations.yml`; pick a profile once with
`hypnose-set-data-location local_1`. Nothing here deletes anything.

In [ ]:
%load_ext autoreload
%autoreload 2

from hypnose_sleap.extract import extract_session
from hypnose_sleap.timestamps import combine_session
from hypnose_sleap.quality import sleap_node_quality_report
from hypnose_sleap.annotate import annotate_session
from hypnose_sleap.io import layout, paths

print("rawdata: ", paths.get_rawdata_root())
print("derivatives:", paths.get_derivatives_root())

## One session, step by step

`extract` reads the session's `.slp` files and writes one parquet per video plus the
quality report; `combine` adds synchronised timestamps and merges them. Pass `model=`
so the quality report records which model produced the predictions -- omitting it
records `UNKNOWN_MODEL` rather than guessing.

In [ ]:
subjid = 57
date = 20260717

extracted = extract_session(subjid, date, model="eeg_headstage", skip_empty=True)
print(f"{len(extracted['outputs'])} per-video table(s)")

In [ ]:
combined = combine_session(subjid, date)
combined.head()

## Tracking quality

Per-node presence and confidence, for comparing models or checking a session before
trusting its centroid. `extract` writes the same numbers to
`movement_analysis/sleap_quality_sub-XXX_ses-YYYYMMDD.yml`.

In [ ]:
quality = sleap_node_quality_report(
    subjid=subjid,
    date=date,
    score_thresh=0.4,
    presence_frac=0.7,
    verbose=True,
)

## Annotated video

Centroid overlay, the active odour near the poke port, and reward markers. Windows are
relative to the video's first frame; each one writes its own `.mp4`. Only `rotate_deg`
0 and 90 are used in practice.

A re-render overwrites a clip of the same name, so give `output_suffix` a distinct
value when you want to keep the previous one.

In [ ]:
outputs = annotate_session(
    subjid, date,
    rotate_deg=90,
    time_window=("00:05:00", "00:07:00"),
    video_indices=[1],       # None renders every video
    reward_display_s=2,
    mark_timepoint=None,     # e.g. "11:33:11.000" to mark a moment
)
outputs

## Inspect SLEAP predictions from a `.slp`

Reads `pred_points` straight out of the HDF5 for a frame range -- the raw predictions,
before any centroid selection. Handles both the structured-flat and dense layouts.

In [ ]:
# Inspect SLEAP predictions from .slp for a frame range
import h5py
import numpy as np
import pandas as pd
import json
from pathlib import Path

# --- Configure here ---
slp_path = r"Z:/hypnose/derivatives/sub-040_id-259/ses-040_date-20251218/saved_analysis_results/2025-12-18T15-59-34__VideoData_1904-01-23T04-00-00.predictions.slp"
frame_start = 26057
frame_end = 26080
use_full_range = False   # set True to ignore frame_start/frame_end and return all frames
node_idx_max = 4        # only include node indices 0..node_idx_max
instance_idx = 0        # dense layout only
# ----------------------

slp_path = Path(slp_path)
if not slp_path.exists():
    raise FileNotFoundError(slp_path)

debug_path = slp_path.parent / debug_csv_name
with h5py.File(slp_path, "r") as f:
    pts = f["pred_points"]
    conf = f.get("pred_confidence")
    frames_ds = f.get("frames")
    node_names = None
    if "node_names" in f:
        try:
            node_names = [n.decode("utf-8") if isinstance(n, (bytes, np.bytes_)) else str(n) for n in f["node_names"][:]]
        except Exception:
            node_names = None

    ndim = pts.ndim
    shape = pts.shape
    dtype_names = list(pts.dtype.names) if pts.dtype.names else []
    print(f"pred_points shape: {shape}")
    if dtype_names:
        print(f"pred_points dtype fields: {dtype_names}")
    if conf is not None:
        print(f"pred_confidence shape: {conf.shape}")

    # Build frame/offset mapping (mimic sleap_utils logic)
    frames_df = None
    video_lookup = {}
    offsets = {}
    if frames_ds is not None and frames_ds.dtype.names:
        frames_fields = list(frames_ds.dtype.names or [])
        frame_num_field = next((n for n in frames_fields if n in ("frame_idx", "frame_number", "frame")), None)
        video_field = next((n for n in frames_fields if n in ("video", "video_id", "video_idx")), None)
        frames_arr = frames_ds[:]
        frames_df = pd.DataFrame({"frame_id": np.arange(len(frames_arr))})
        frames_df["frame_local"] = frames_arr[frame_num_field].astype(int) if frame_num_field else frames_df["frame_id"]
        frames_df["video_idx"] = frames_arr[video_field].astype(int) if video_field else 0
        # Video names from videos_json if present
        vids = None
        if "videos_json" in f:
            try:
                vids_raw = f["videos_json"][()]
                vids = json.loads(vids_raw.decode("utf-8") if isinstance(vids_raw, (bytes, np.bytes_)) else vids_raw)
            except Exception:
                vids = None
        if isinstance(vids, list):
            for i, v in enumerate(vids):
                name = None
                if isinstance(v, dict):
                    name = v.get("filename") or v.get("file")
                if not name and isinstance(v, str):
                    name = v
                if name:
                    video_lookup[i] = Path(name).name
        # Offsets per video (ordered by video_idx) for global frame numbering
        total = 0
        for vid in sorted(frames_df["video_idx"].unique()):
            offsets[vid] = total
            sub = frames_df[frames_df["video_idx"] == vid]
            if not sub.empty and sub["frame_local"].notna().any():
                total += int(sub["frame_local"].max()) + 1
            else:
                total += len(sub)
        print(f"frames dataset fields: {frames_fields}")
        print(f"video offsets: {offsets}")

    df_slice = pd.DataFrame()
    structured_handled = False
    # Structured flat layout: pred_points is 1-D with fields like x/y/score and instances reference slices.
    if ndim == 1 and dtype_names and {"x", "y"}.issubset(dtype_names):
        inst_ds = f.get("instances")
        if inst_ds is None:
            raise ValueError("Structured pred_points but instances dataset missing; cannot map to frames")
        inst_fields = list(inst_ds.dtype.names or [])
        frame_field = next((n for n in inst_fields if n in ("frame_id", "frame", "frame_idx")), None)
        start_field = next((n for n in inst_fields if "point_id_start" in n or ("point" in n and "start" in n)), None)
        end_field = next((n for n in inst_fields if "point_id_end" in n or ("point" in n and "end" in n)), None)
        if not all([frame_field, start_field, end_field]):
            print(f"instances dtype fields: {inst_fields}")
            raise ValueError("Could not find frame/start/end fields in instances dataset")

        inst_arr = inst_ds[:]
        inst_df = pd.DataFrame({
            "frame_id": inst_arr[frame_field].astype(int),
            "start": inst_arr[start_field].astype(int),
            "end": inst_arr[end_field].astype(int),
            "instance_id": inst_arr["instance_id"].astype(int) if "instance_id" in inst_fields else np.arange(len(inst_arr)),
        })
        inst_df["count"] = inst_df["end"] - inst_df["start"]
        if frames_df is not None:
            inst_df = inst_df.join(frames_df.set_index("frame_id"), on="frame_id", how="left")
            inst_df["video_file"] = inst_df["video_idx"].map(video_lookup) if video_lookup else None
            inst_df["global_frame"] = inst_df["frame_local"] + inst_df["video_idx"].map(offsets)
        # Filter range using global_frame if present, else frame_local, else frame_id
        if use_full_range:
            mask = inst_df["count"] > 0
        elif "global_frame" in inst_df.columns and inst_df["global_frame"].notna().any():
            mask = (inst_df["global_frame"] >= frame_start) & (inst_df["global_frame"] <= frame_end) & (inst_df["count"] > 0)
        elif "frame_local" in inst_df.columns and inst_df["frame_local"].notna().any():
            mask = (inst_df["frame_local"] >= frame_start) & (inst_df["frame_local"] <= frame_end) & (inst_df["count"] > 0)
        else:
            mask = (inst_df["frame_id"] >= frame_start) & (inst_df["frame_id"] <= frame_end) & (inst_df["count"] > 0)
        inst_df = inst_df[mask]
        if inst_df.empty:
            print("No instances in requested frame range")
        else:
            rows = []
            for _, inst in inst_df.reset_index(drop=True).iterrows():
                start = int(inst["start"])
                end = int(inst["end"])
                pt_slice = pts[start:end]
                for node_idx, pt in enumerate(pt_slice):
                    if node_idx_max is not None and node_idx > node_idx_max:
                        break
                    rows.append({
                        "frame": int(inst.get("global_frame", inst.get("frame_local", inst.get("frame_id", 0)))),
                        "node_idx": int(node_idx),
                        "x": float(pt["x"]),
                        "y": float(pt["y"]),
                        "score": float(pt["score"]) if "score" in dtype_names else np.nan,
                    })
            df_slice = pd.DataFrame(rows, columns=["frame", "node_idx", "x", "y", "score"])
            print(f"Structured layout rows: {len(df_slice)}")
            structured_handled = True

    if structured_handled:
        pass
    else:
        # Guard against unexpected storage layouts
        if ndim not in (3, 4):
            print("Unexpected pred_points ndim; listing file keys for inspection:")
            print(list(f.keys()))
            try:
                arr_sample = pts[: min(5, pts.shape[0])]
                print(f"Sample dtype: {arr_sample.dtype}, sample shape: {arr_sample.shape}")
            except Exception as e:
                print(f"Could not sample pred_points: {e}")
            raise ValueError(f"Unexpected pred_points ndim={ndim}")

        # Slice frames (dense layout)
        fs = 0 if use_full_range else max(frame_start, 0)
        fe = shape[0] - 1 if use_full_range else min(frame_end, shape[0] - 1)
        if fe < fs:
            raise ValueError("frame_end before frame_start or out of range")

        if ndim == 4:  # (frames, instances, nodes, 2)
            pts_slice = pts[fs:fe+1, instance_idx, :, :]
            conf_slice = conf[fs:fe+1, instance_idx, :] if conf is not None else None
            nodes = pts_slice.shape[1]
        elif ndim == 3:  # (frames, nodes, 2)
            pts_slice = pts[fs:fe+1, :, :]
            conf_slice = conf[fs:fe+1, :] if conf is not None else None
            nodes = pts_slice.shape[1]
        max_nodes = min(nodes, node_idx_max + 1) if node_idx_max is not None else nodes

        # Build rows (long form)
        rows = []
        frame_vals = np.arange(fs, fe + 1)
        for fi, frame in enumerate(frame_vals):
            for ni in range(max_nodes):
                score_val = float(conf_slice[fi, ni]) if conf_slice is not None else np.nan
                rows.append({
                    "frame": int(frame),
                    "node_idx": int(ni),
                    "x": float(pts_slice[fi, ni, 0]),
                    "y": float(pts_slice[fi, ni, 1]),
                    "score": score_val,
                })
        df_slice = pd.DataFrame(rows, columns=["frame", "node_idx", "x", "y", "score"])
        print(f"Dense layout frames {fs}-{fe}, instance {instance_idx}, nodes {max_nodes}")

    if not df_slice.empty:
        df_slice.to_csv(debug_path, index=False)
        print(f"Saved debug CSV: {debug_path}")
        print(df_slice.head(20))
    else:
        print("No data to save")